# Generate cn data and save to './data/{filename}.parquet'

In [1]:
from src.generate_data import generate_cn_european_data, generate_cn_american_data
import cProfile
import pstats

european_versions = ("v6", "v7")
s_steps = [100, 500] + list(range(1000, 5000, 1000))
s_steps = tuple(s_steps)
t_steps = [100, 500] + list(range(1000, 5000, 1000))
t_steps = tuple(t_steps)
n_reps = 5  # for the entire run
warmup_runs = 3 # per method

# european puts just use put call parity
GENERATE_EUROPEAN_DATA = True
GENERATE_AMERICAN_CALL_DATA = True
GENERATE_AMERICAN_PUT_DATA = True



# american data always generates calls and puts

K = 100
S0 = 100
T = 3
r = 0.02
sigma = 0.4

n_total_runs = 0

if GENERATE_EUROPEAN_DATA: n_total_runs += len(s_steps) * len(t_steps) * len(european_versions) * n_reps
if GENERATE_AMERICAN_CALL_DATA: n_total_runs += len(s_steps) * len(t_steps) * n_reps
if GENERATE_AMERICAN_PUT_DATA: n_total_runs += len(s_steps) * len(t_steps) * n_reps

# grid uses x = logS
print(f"executing {n_total_runs} runs in total")
print(f"for x grid steps in: {s_steps}")
print(f"for t grid steps in: {t_steps}")
print(f"repeating for {n_reps} repetitions")

executing 720 runs in total
for x grid steps in: (100, 500, 1000, 2000, 3000, 4000)
for t grid steps in: (100, 500, 1000, 2000, 3000, 4000)
repeating for 5 repetitions


In [2]:
print("performing warmup cn runs..")
from src.options import EuropeanOption, AmericanOption

options = []
if GENERATE_EUROPEAN_DATA: options.append(EuropeanOption(100, 3, contract_type='call'))
if GENERATE_AMERICAN_CALL_DATA: options.append(AmericanOption(100, 3, contract_type='call'))
if GENERATE_AMERICAN_PUT_DATA: options.append(AmericanOption(100, 3, contract_type='put'))


for option in options:
    for i in range(warmup_runs):
        if isinstance(option, EuropeanOption):
            for v in european_versions:
                option.price_CN(S0=S0, r=r, sigma=sigma, s_steps=500, t_steps=500, version=v)
        else:
            option.price_CN(S0=S0, r=r, sigma=sigma, s_steps=500, t_steps=500)


performing warmup cn runs..


In [3]:
n_reps = 20
cn_config = dict(
    K = K,
    T = T,
    S0 = S0,
    r = r,
    sigma = sigma,
    n_reps = n_reps,
    s_steps = s_steps,
    t_steps = t_steps,
)





# generate data

if GENERATE_EUROPEAN_DATA:
    filename = "profiled_cn_run_eu"
    print("\n executing european option runs")
    profiler = cProfile.Profile()
    profiler.enable()
    df_european_cn = generate_cn_european_data(**cn_config, filename=filename,versions=european_versions, save=True)
    profiler.disable()
    eu_stats = pstats.Stats(profiler)
    eu_stats.strip_dirs()
    eu_stats.sort_stats("cumulative")
    profiler.dump_stats(f'./data/{filename}.prof')
    eu_stats.print_stats(20)

if GENERATE_AMERICAN_PUT_DATA or GENERATE_AMERICAN_CALL_DATA:
    filename = "profiled_cn_run_am"
    print("\n executing american option runs")
    profiler = cProfile.Profile()
    profiler.enable()
    df_american_cn = generate_cn_american_data(**cn_config, filename=filename, save=True)
    profiler.disable()
    am_stats = pstats.Stats(profiler)
    am_stats.strip_dirs()
    am_stats.sort_stats("cumulative")
    profiler.dump_stats(f'./data/{filename}.prof')
    am_stats.print_stats(20)



 executing european option runs
versions chosen: ('v6', 'v7')


Running:   0%|          | 0/1440 [00:00<?, ?run/s]

         335286 function calls (332022 primitive calls) in 21.125 seconds

   Ordered by: cumulative time
   List reduced from 1178 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
  423/422    0.005    0.000   43.859    0.104 base_events.py:1962(_run_once)
      422    0.006    0.000   27.622    0.065 selectors.py:310(select)
     1440    0.025    0.000   18.593    0.013 options.py:69(price_CN)
      720   12.733    0.018   12.765    0.018 fdm.py:672(pde_crank_nicolson_v6)
      423    0.006    0.000   12.091    0.029 events.py:87(_run)
      720    5.850    0.008    5.880    0.008 fdm.py:802(pde_crank_nicolson_v7)
      423    0.001    0.000    2.927    0.007 {method 'run' of '_contextvars.Context' objects}
      413    0.002    0.000    2.220    0.005 iostream.py:351(<lambda>)
      418    0.002    0.000    2.118    0.005 zmqstream.py:573(_handle_events)
3308/3154    1.602    0.000    2.113    0.001 socket.py:623(send)
      418 

Running:   0%|          | 0/1440 [00:00<?, ?run/s]

         276025 function calls (273204 primitive calls) in 13.439 seconds

   Ordered by: cumulative time
   List reduced from 818 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     1440   11.999    0.008   12.059    0.008 fdm.py:913(pde_crank_nicolson_american)
     1440    0.024    0.000   11.882    0.008 options.py:148(price_CN)
      331    0.007    0.000    9.138    0.028 events.py:87(_run)
      328    0.004    0.000    3.217    0.010 selectors.py:310(select)
      331    0.001    0.000    1.870    0.006 {method 'run' of '_contextvars.Context' objects}
      320    0.002    0.000    1.510    0.005 iostream.py:351(<lambda>)
      326    0.001    0.000    1.440    0.004 zmqstream.py:573(_handle_events)
      325    0.001    0.000    1.422    0.004 zmqstream.py:614(_handle_recv)
      325    0.001    0.000    1.396    0.004 zmqstream.py:546(_run_callback)
2564/2344    0.927    0.000    1.292    0.001 socket.py:623(send)
      